# IBM Default Probability: Merton, KMV-Style Iteration and the Naive Model

How far is IBM from default, and how does the answer depend on the model? The notebook estimates IBM's one-year distance to default (DD) and default probability (PD) at each month-end of the last year from market data:

1. **Merton (1974) with the KMV iterative procedure:** treat equity as a call on the firm's assets with strike equal to the default point, and iterate until the asset volatility implied by the daily asset values matches the one used to back them out.
2. **Merton with drift:** the same assets, but the real-world PD uses the estimated asset drift μ instead of the risk-free rate.
3. **Naive model (Bharath and Shumway, 2008):** no equation solving; asset volatility is approximated from equity volatility and a debt-volatility rule of thumb.

Each estimate uses only the 252 trading days up to its date. Asset values are priced with the risk-free rate; the drift enters only the real-world DD.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.optimize import brentq
from scipy.stats import norm

sns.set_theme(style="whitegrid")
T = 1.0              # horizon, years
WINDOW = 252         # trading days per estimation window

## Data

From Yahoo Finance: IBM daily closes and shares outstanding, IBM's quarterly balance sheets (current and long-term debt), and the 13-week Treasury bill yield (`^IRX`) as the risk-free rate. The first run downloads them; the snapshot in `data/` keeps later runs identical.

In [ ]:
DATA = Path("data")
files = {name: DATA / f"{name}.csv" for name in ["prices", "debt", "rates"]}
if not all(f.exists() for f in files.values()):
    import yfinance as yf
    DATA.mkdir(exist_ok=True)
    ibm = yf.Ticker("IBM")
    hist = ibm.history(period="3y", auto_adjust=False)[["Close"]]
    hist.index = hist.index.tz_localize(None).normalize()
    shares = ibm.get_shares_full(start=hist.index[0])
    shares.index = shares.index.tz_localize(None).normalize()
    shares = shares.groupby(level=0).last()
    hist["shares"] = shares.reindex(hist.index, method="ffill").bfill()
    hist.to_csv(files["prices"])

    sheets = pd.concat([ibm.quarterly_balance_sheet, ibm.balance_sheet], axis=1)
    sheets = sheets.loc[:, ~sheets.columns.duplicated()]
    def row(*names):
        for n in names:
            if n in sheets.index:
                return sheets.loc[n]
        raise KeyError(names)
    debt = pd.DataFrame({
        "current_debt": row("Current Debt", "Current Debt And Capital Lease Obligation"),
        "long_term_debt": row("Long Term Debt", "Long Term Debt And Capital Lease Obligation"),
    }).sort_index().dropna()
    debt.to_csv(files["debt"])

    irx = yf.Ticker("^IRX").history(period="3y")[["Close"]].rename(columns={"Close": "rate_pct"})
    irx.index = irx.index.tz_localize(None).normalize()
    irx.to_csv(files["rates"])

prices = pd.read_csv(files["prices"], index_col=0, parse_dates=True)
debt = pd.read_csv(files["debt"], index_col=0, parse_dates=True)
rates = pd.read_csv(files["rates"], index_col=0, parse_dates=True)
debt

The **default point** is the KMV convention: short-term debt plus half of long-term debt, in dollars. Each balance sheet applies from its date until the next one; days before the earliest available sheet use that sheet.

In [ ]:
df = prices.join(rates, how="inner")
df["E"] = df["Close"] * df["shares"]
df["r"] = np.log1p(df["rate_pct"] / 100)          # continuously compounded
default_point = debt["current_debt"] + 0.5 * debt["long_term_debt"]
df["F"] = default_point.reindex(df.index, method="ffill").bfill()
df = df.dropna()

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
(df["E"] / 1e9).plot(ax=axes[0], title="Market value of equity ($bn)")
(df["F"] / 1e9).plot(ax=axes[1], title="Default point ($bn)")
(100 * df["r"]).plot(ax=axes[2], title="Risk-free rate (%)")
plt.tight_layout()
plt.show()
df[["E", "F", "r"]].describe().T

## The models

In [ ]:
def d1_d2(V, F, r, sigma, T=T):
    d1 = (np.log(V / F) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    return d1, d1 - sigma * np.sqrt(T)

def equity_value(V, F, r, sigma):
    d1, d2 = d1_d2(V, F, r, sigma)
    return V * norm.cdf(d1) - F * np.exp(-r * T) * norm.cdf(d2)

def implied_assets(E, F, r, sigma):
    """Asset value V solving E = BS call(V, F, r, sigma), day by day."""
    return np.array([brentq(lambda v: equity_value(v, f, rr, sigma) - e, e, e + 2 * f)
                     for e, f, rr in zip(E, F, r)])

def kmv(window, tol=1e-6, max_iter=100):
    """Iterate sigma_V until the volatility of the implied asset path reproduces it."""
    E, F, r = window["E"].to_numpy(), window["F"].to_numpy(), window["r"].to_numpy()
    sigma_E = np.std(np.diff(np.log(E)), ddof=1) * np.sqrt(252)
    sigma = sigma_E * E[-1] / (E[-1] + F[-1])
    for i in range(max_iter):
        V = implied_assets(E, F, r, sigma)
        new_sigma = np.std(np.diff(np.log(V)), ddof=1) * np.sqrt(252)
        if abs(new_sigma - sigma) < tol:
            break
        sigma = new_sigma
    rets = np.diff(np.log(V))
    mu = rets.mean() * 252 + 0.5 * new_sigma**2        # annual drift of assets
    return {"V": V[-1], "sigma_V": new_sigma, "mu": mu, "sigma_E": sigma_E, "iterations": i + 1}

def distance_to_default(V, F, drift, sigma):
    return (np.log(V / F) + (drift - 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))

def naive(window):
    """Bharath and Shumway (2008): sigma_D = 0.05 + 0.25 sigma_E, V = E + F, mu = past equity return."""
    E, F = window["E"].iloc[-1], window["F"].iloc[-1]
    sigma_E = np.std(np.diff(np.log(window["E"])), ddof=1) * np.sqrt(252)
    sigma_D = 0.05 + 0.25 * sigma_E
    sigma_V = E / (E + F) * sigma_E + F / (E + F) * sigma_D
    mu = np.log(window["E"].iloc[-1] / window["E"].iloc[0])
    return distance_to_default(E + F, F, mu, sigma_V), sigma_V

## Month-end estimates over the last year

In [ ]:
month_ends = df.groupby(df.index.to_period("M")).tail(1).index
month_ends = [d for d in month_ends if df.index.get_loc(d) >= WINDOW][-12:]

rows = []
for date in month_ends:
    w = df.loc[:date].iloc[-WINDOW:]
    est = kmv(w)
    F, r = w["F"].iloc[-1], w["r"].iloc[-1]
    dd_rn = distance_to_default(est["V"], F, r, est["sigma_V"])
    dd_mu = distance_to_default(est["V"], F, est["mu"], est["sigma_V"])
    dd_naive, sigma_naive = naive(w)
    rows.append({"date": date.date(), "E ($bn)": w["E"].iloc[-1] / 1e9, "F ($bn)": F / 1e9,
                 "V ($bn)": est["V"] / 1e9, "sigma_E": est["sigma_E"], "sigma_V": est["sigma_V"],
                 "mu": est["mu"], "iterations": est["iterations"],
                 "DD risk-neutral": dd_rn, "DD with drift": dd_mu, "DD naive": dd_naive,
                 "PD risk-neutral": norm.cdf(-dd_rn), "PD with drift": norm.cdf(-dd_mu),
                 "PD naive": norm.cdf(-dd_naive), "sigma_V naive": sigma_naive})
results = pd.DataFrame(rows).set_index("date")
results.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))
results[["DD risk-neutral", "DD with drift", "DD naive"]].plot(ax=axes[0], marker="o", title="Distance to default")
results[["sigma_E", "sigma_V", "sigma_V naive"]].plot(ax=axes[1], marker="o", title="Equity and asset volatility")
plt.tight_layout()
plt.show()
results[["PD risk-neutral", "PD with drift", "PD naive"]].map(lambda p: f"{p:.2e}")

In [ ]:
# Checks: the iteration converged, assets price equity back, and V exceeds equity.
last = df.loc[:month_ends[-1]].iloc[-WINDOW:]
est = kmv(last)
assert est["iterations"] < 100
assert np.isclose(equity_value(est["V"], last["F"].iloc[-1], last["r"].iloc[-1], est["sigma_V"]), last["E"].iloc[-1], rtol=1e-6)
assert (results["V ($bn)"] > results["E ($bn)"]).all()
assert (results["sigma_V"] < results["sigma_E"]).all()

## Conclusion

CONCLUSION_PLACEHOLDER